In [ ]:
import mysql.connector
from faker import Faker
import random

# --- CONFIGURATION ---
db_config = {
    'host': '***********',
    'user': '******',
    'password': '**********', # PUT YOUR PASSWORD HERE IF YOU HAVE ONE
    'database': 'ecommerce_db'
}

fake = Faker()

# --- CONNECT ---
def get_connection():
    return mysql.connector.connect(**db_config)

try:
    conn = get_connection()
    cursor = conn.cursor()
    print("Successfully connected to the database!")
except Exception as e:
    print(f"Connection Failed: {e}")

# --- DATA GENERATION FUNCTIONS ---

def create_categories():
    categories = [
        ('Electronics', 'Gadgets, computers, and accessories'),
        ('Clothing', 'Men and Women fashion'),
        ('Home & Kitchen', 'Appliances and decor'),
        ('Books', 'Fiction, non-fiction, and educational'),
        ('Sports', 'Gym equipment and outdoor gear')
    ]
    sql = "INSERT IGNORE INTO categories (category_name, description) VALUES (%s, %s)"
    cursor.executemany(sql, categories)
    conn.commit()
    print(f"Categories ready.")

def create_products():
    cursor.execute("SELECT category_id FROM categories")
    cat_ids = [row[0] for row in cursor.fetchall()]
    
    if not cat_ids:
        print("No categories found! Cannot create products.")
        return

    products = []
    for _ in range(50): 
        cat_id = random.choice(cat_ids)
        name = fake.word().capitalize() + " " + fake.word().capitalize()
        price = round(random.uniform(10, 1000), 2)
        stock = random.randint(10, 100)
        products.append((cat_id, name, price, stock))
    
    sql = "INSERT INTO products (category_id, product_name, price, stock_quantity) VALUES (%s, %s, %s, %s)"
    cursor.executemany(sql, products)
    conn.commit()
    print(f"Inserted {len(products)} Products")

def create_users():
    users = []
    existing_emails = set()
    
    for _ in range(50): 
        first = fake.first_name()
        last = fake.last_name()
        email = f"{first.lower()}.{last.lower()}{random.randint(1,999)}@example.com"
        
        if email not in existing_emails:
            users.append((first, last, email, fake.password(), fake.phone_number(), fake.city()))
            existing_emails.add(email)
    
    sql = "INSERT IGNORE INTO users (first_name, last_name, email, password_hash, phone_number, city) VALUES (%s, %s, %s, %s, %s, %s)"
    cursor.executemany(sql, users)
    conn.commit()
    print(f"Users ready (Attempts: {len(users)})")

def create_orders_and_items():
    # 1. Get Users and Products
    cursor.execute("SELECT user_id FROM users")
    user_ids = [row[0] for row in cursor.fetchall()]
    
    cursor.execute("SELECT product_id, price FROM products")
    product_data = cursor.fetchall()
    
    if not user_ids or not product_data:
        print("Error: Missing users or products.")
        return

    orders_created = 0
    
    for _ in range(150): 
        user_id = random.choice(user_ids)
        # Generate date object
        date_obj = fake.date_time_between(start_date='-1y', end_date='now')
        # Convert to string to avoid MySQL Timestamp issues
        order_date_str = date_obj.strftime('%Y-%m-%d %H:%M:%S')
        
        status = random.choice(['Pending', 'Completed', 'Shipped', 'Cancelled'])
        
        # Create Order
        cursor.execute("INSERT INTO orders (user_id, order_date, status, total_amount) VALUES (%s, %s, %s, %s)", 
                       (user_id, order_date_str, status, 0))
        order_id = cursor.lastrowid
        
        # Add Items
        num_items = random.randint(1, 5)
        total_amount = 0
        
        for _ in range(num_items):
            prod = random.choice(product_data)
            p_id, p_price = prod[0], float(prod[1])
            qty = random.randint(1, 3)
            
            cursor.execute("INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES (%s, %s, %s, %s)",
                           (order_id, p_id, qty, p_price))
            total_amount += (p_price * qty)
            
        # Update Total
        cursor.execute("UPDATE orders SET total_amount = %s WHERE order_id = %s", (total_amount, order_id))
        
        # Create Payment
        if status in ['Completed', 'Shipped']:
            pay_method = random.choice(['Credit Card', 'PayPal', 'Bank Transfer'])
            cursor.execute("INSERT INTO payments (order_id, payment_date, payment_method, amount, status) VALUES (%s, %s, %s, %s, %s)",
                           (order_id, order_date_str, pay_method, total_amount, 'Success'))
        
        orders_created += 1

    conn.commit()
    print(f"Inserted {orders_created} Orders with Items and Payments")

# --- RUN ---
if 'conn' in locals() and conn.is_connected():
    # Clear old data to start fresh
    cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
    cursor.execute("TRUNCATE TABLE payments")
    cursor.execute("TRUNCATE TABLE order_items")
    cursor.execute("TRUNCATE TABLE orders")
    cursor.execute("TRUNCATE TABLE products")
    cursor.execute("TRUNCATE TABLE categories")
    cursor.execute("TRUNCATE TABLE users")
    cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
    print("Cleared old data.")

    create_categories()
    create_products()
    create_users()
    create_orders_and_items()
    
    cursor.close()
    conn.close()
    print("\nDATA GENERATION COMPLETE!")

Successfully connected to the database!
Cleared old data.
Categories ready.
Inserted 50 Products
Users ready (Attempts: 50)
Inserted 150 Orders with Items and Payments

DATA GENERATION COMPLETE!
